In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "global_settings.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
from src.global_settings import settings


In [ ]:
from pathlib import Path
from dataclasses import asdict

import pandas as pd
from transformers import AutoTokenizer

from src.global_settings import settings
from src.features import load_score_data, load_sample_records
from src.auto_explainer.prompts import (
    SYSTEM_MESSAGE,
    Role,
    PromptBuilder,
    TermLevel,
    format_user_message,
    format_sample_records,
)
from src.auto_explainer.explainer import ZeroShotAutoExplainer
from src.utils.vllm.client import VLLMClient
from src.utils.vllm.config import GenConfig

In [ ]:
META_DIR = Path(settings.feature_meta_dir)
RECORD_DIR = Path(settings.feature_record_dir)
EXPLANATION_DIR = Path(settings.feature_explanation_dir)

## 1. Load features to explain

In [ ]:
MODEL = "f3072_k8"
_HPARAM = "alpha"
LAYER = 21
CHECKPOINT = f"{MODEL}_{_HPARAM}"

### 1.1 filter major features (the feature that is 'alive' and has at least 200 samples)

In [ ]:
scores_all = load_score_data(checkpoint=CHECKPOINT, layer=LAYER, metric_name="total_post")
scores_all_df = pd.DataFrame(scores_all)

active_sample_counts = (scores_all_df > 0).sum(axis=0)
active_sample_counts = active_sample_counts[active_sample_counts > 0]
print(len(active_sample_counts))

In [ ]:
# # validation: check if the sampel-record is empty or zero-fulled
# actual_live_features = get_available_features(checkpoint=CHECKPOINT, layer=LAYER)
# print(len(actual_live_features))

# if len(active_sample_counts) == len(actual_live_features):
#     print("✅")
# else:
#     print("❌")

**요렇게 major feature 인덱스 리스트를 얻게 됨. 요걸 쓰자.**

In [ ]:
MIN_ACTIVE_SAMPLES = 200

major_features_indices = active_sample_counts[active_sample_counts > MIN_ACTIVE_SAMPLES].index.tolist()

print("Major Features", len(major_features_indices))
print(f"{major_features_indices[:5]}", "...")

In [ ]:
for i, feat_idx in enumerate(major_features_indices):
    pos_samples = load_sample_records(CHECKPOINT, LAYER, feature_index=feat_idx, sample_type="positive", as_frame=False)
    neg_samples = load_sample_records(CHECKPOINT, LAYER, feature_index=feat_idx, sample_type="negative", as_frame=False)
    
    print(f"Feature {feat_idx}: +{len(pos_samples)} / -{len(neg_samples)} samples")

    if i >= 5:
        break

print("...")

| use | drop |
| --- | ---- |
| text, active_tokens | text_idx, label, pred, model, prompt, total_score_post, avg_score_pre |

In [ ]:
pos_samples = load_sample_records(CHECKPOINT, LAYER, feature_index=3047, sample_type="positive", as_frame=False)

formatted_sample_records = format_sample_records(samples=[asdict(pos_samples[0]), asdict(pos_samples[1])], num_samples=2, topk_tokens=5, order_tokens=False, omit_zeros=True)

print(formatted_sample_records)

In [ ]:
sample_record = asdict(pos_samples[0])
print(sample_record.keys())

sample_text = sample_record['text']
sample_tokens = sample_record["active_tokens"]
print(sample_tokens)

## 2. Check Prompt Token Length

In [ ]:
llm_ckpt = "Qwen/Qwen3-30B-A3B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(llm_ckpt)

In [ ]:
MAX_TOKENS = 32768

def calculate_token_count(text: str, feat_idx: int) -> int:
    tokens = tokenizer.tokenize(text)
    if len(tokens) > MAX_TOKENS:
        print(f"Feature #{feat_idx} - Total Tokens: {len(tokens):,} ({round(len(tokens)/MAX_TOKENS*100)}%) ❌")
    else:
        print(f"Feature #{feat_idx} - Total Tokens: {len(tokens):,} ({round(len(tokens)/MAX_TOKENS*100)}%) ✅")

In [ ]:
feature_indices = major_features_indices

for feature_index in feature_indices:
    sample_records = load_sample_records(RECORD_DIR / CHECKPOINT / str(LAYER), feature_index=feature_index, sample_type="positive", as_frame=False)
    samples = [asdict(sample_record) for sample_record in sample_records]
    user_message = format_user_message(feature_index=feature_index, samples=samples, num_samples=20, topk_tokens=10)
    
    calculate_token_count(user_message, feature_index)

## 3. VLLMClient 세팅

In [ ]:
llm_client = VLLMClient(async_mode=True)

In [ ]:
gen_config = GenConfig(
    temperature=0.1,
    top_p=0.9,
    max_tokens=1024,
)

In [ ]:
print(f"Current serving model is: {llm_client.model_id}")
print(f"Current serving model config: ")
model_config = asdict(llm_client.model_config)
for k, v in model_config.items():
    print(f"\t - {k}: {v}")

In [ ]:
print(f"Default generation config: ")
for k, v in asdict(gen_config).items():
    print(f"\t - {k}: {v}")

## 4. Generate

In [ ]:
explainer = ZeroShotAutoExplainer(llm_client=llm_client)

**문장 단위의 설명 생성:**

In [ ]:
feature_index = major_features_indices[1]
print("Seletected feature: ", feature_index)

explainer.init_feature(checkpoint=CHECKPOINT, layer=LAYER, feature_index=feature_index)
explanation_term = explainer.generate_explanation(gen_config=gen_config, term_level=TermLevel.PHRASE, num_samples=30, num_tokens=5)

print("Explanation: ", explanation_term)

**구문 단위의 설명 생성:**

In [ ]:
feature_index = major_features_indices[1]
print("Seletected feature: ", feature_index)

explainer.init_feature(checkpoint=CHECKPOINT, layer=LAYER, feature_index=feature_index)
explanation_sent = explainer.generate_explanation(gen_config=gen_config, term_level=TermLevel.SENTENCE, num_samples=20, num_tokens=10)

print("Explanation: ", explanation_sent)

**생성된 설명 결과 저장:** `data/feature_explanations/checkpoint/layer/{feature_index}.jsonl`에 jsonl 형식으로 저장

In [ ]:
explainer.save_unscored_explanations(explanations=[explanation_term, explanation_sent])

### 99. Explainer 거치지 않고 llm_client 그대로 사용해서 생성하는 방식(deprecated, 참고 only)

In [ ]:
feature_explanations = {}

for feature_index in feature_indices:
    prompt_builder = PromptBuilder()
    prompt_builder.add_message(role=Role.SYSTEM, content=SYSTEM_MESSAGE)

    sample_records = load_sample_records(RECORD_DIR / CHECKPOINT / str(LAYER), feature_index=feature_index, sample_type="positive", as_frame=False)
    samples = [asdict(sample_record) for sample_record in sample_records]
    user_message = format_user_message(feature_index=feature_index, samples=samples, num_samples=20, topk_tokens=10)

    prompt_builder.add_message(role=Role.USER, content=user_message)
    # prompt = prompt_builder.build(prompt_format=PromptFormat.CHAT)

    response = llm_client.generate(prompt=user_message, config=gen_config, system_prompt=SYSTEM_MESSAGE, extract_reasoning=False)
    explanation = response

    feature_explanations[feature_index] = explanation

    print(f"Feature: #{feature_index}: {explanation}\n")